# 03. CapL Project — 공동 Problem Solving Notebook

고려대학교–SK하이닉스 산학 교육 프로그램 · **In-Line ICBV 학습을 통한 CapL 예측 Model 개발** · 9/23

이 Notebook은 실제 데이터를 함께 분석하고 현업 의견을 다음 분석으로 연결하는 기록장입니다.

**Apply → Analyze → Discuss → Domain Knowledge → Improve → Re-Analyze → Next Hypothesis**

## Section 0. Today's Question

**현재 제공된 데이터를 이용하여 ICBV 및 공정 Context와 CapL 사이에서 어떤 패턴을 발견할 수 있는가?**

- 분석 일자 / 참여자:
- 한 행의 의미(예: 측정 단위, 집계 단위):
- ICBV / CapL 정의와 단위:
- 측정 시점·매칭 키·중복 관측 여부:



## Section 1. Data Loading — Apply

실제 CSV 위치와 실제 열 이름을 아래에 지정하세요. 상대 경로는 프로젝트 루트(`hynix-iTAP`) 기준입니다.
원본 CSV를 프로젝트 루트의 `data/private/`에 두거나 절대 경로를 사용할 수 있습니다. 
`column_map`은 **분석 역할 → 실제 열 이름**입니다. 없는 Context는 `None`으로 설정합니다.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 실제 데이터 수령 후 해당 경로만 변경
DATA_PATH = Path("data/private/capl_process_data.csv")
CSV_ENCODING = "utf-8-sig"
CSV_SEPARATOR = ","
column_map = {
    "ICBV": "ICBV", "CapL": "CapL", "PRODUCT": "PRODUCT",
    "EQUIPMENT": "EQUIPMENT", "CHAMBER": None, "RECIPE": "RECIPE",
}

In [2]:
# 실행 위치와 관계없이 requirements.txt와 week1이 있는 프로젝트 루트를 찾습니다.
current_dir = Path.cwd()
project_root = next(
    (folder for folder in [current_dir, *current_dir.parents]
     if (folder / "requirements.txt").is_file() and (folder / "week1").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("교재 폴더 전체를 내려받고 hynix-iTAP 안에서 실행하세요.")

In [3]:
# 상대 경로는 공통 data 폴더가 있는 프로젝트 루트 기준입니다.
csv_path = DATA_PATH if DATA_PATH.is_absolute() else project_root / DATA_PATH
df = None
if csv_path.is_file():
    try:
        df = pd.read_csv(csv_path, encoding=CSV_ENCODING, sep=CSV_SEPARATOR)
        print("실제 CSV 로드 완료:", df.shape)
    except (OSError, UnicodeError, pd.errors.ParserError, pd.errors.EmptyDataError) as error:
        print("CSV를 읽지 못했습니다. 경로·인코딩·구분자를 확인하세요.")
        print(type(error).__name__)
else:
    print("실제 데이터가 아직 없습니다. DATA_PATH를 지정한 뒤 위에서부터 재실행하세요.")

실제 데이터가 아직 없습니다. DATA_PATH를 지정한 뒤 위에서부터 재실행하세요.


## Section 2. Understand Data — Analyze

**왜?** 열 이름이나 자료형을 추측하면 잘못된 비교를 할 수 있습니다.
아래 출력에서 실제 열 이름을 확인한 뒤 `column_map`을 수정하고 재실행하세요.

In [4]:
numeric_columns = []
categorical_columns = []
if df is None:
    print("데이터 대기 중: 구조 확인을 건너뜁니다.")
else:
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    display(df.head())
    df.info()
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str), "missing": df.isnull().sum(),
        "unique_count": df.nunique(),
    }))
    numeric_columns = df.select_dtypes(include="number").columns.tolist()
    categorical_columns = df.select_dtypes(exclude="number").columns.tolist()
    print("수치 dtype:", numeric_columns)
    print("비수치 dtype:", categorical_columns)

데이터 대기 중: 구조 확인을 건너뜁니다.


In [5]:
if df is not None:
    for role, column in column_map.items():
        if column is None or column not in df.columns:
            print(f"{role}: 미설정 또는 없는 열 → 해당 분석 생략")
        else:
            # 값 전체 대신 앞의 10종류만 확인합니다.
            print(role, "→", column, "| unique 예시:", df[column].dropna().unique()[:10])
    print("완전 중복 행 수:", df.duplicated().sum())

**함께 확인:** 숫자 ID도 의미상 범주형일 수 있습니다. 중복 행은 반복 측정인지 오류인지 확인한 뒤 처리합니다.
숫자가 문자열로 읽혔다면 단위 문자·구분자·특수 결측 표기를 먼저 확인하세요. 아래 함수는 숫자형 측정값만 사용합니다.
양/음의 무한대는 분석용 복사본에서만 제외하고 개수를 알려줍니다. 원본 `df`는 유지합니다.

In [6]:
def numeric_measurement(data, role):
    # 역할에 연결된 열이 실제 숫자형일 때만 분석합니다.
    column = column_map.get(role)
    if data is None or column is None or column not in data.columns:
        print(f"{role}: 데이터 또는 매핑된 열이 없어 생략합니다.")
        return None
    if not pd.api.types.is_numeric_dtype(data[column]) or pd.api.types.is_bool_dtype(data[column]):
        print(f"{role}: 숫자형 측정 열이 아닙니다. 원본 형식을 확인하세요.")
        return None
    values = data[column].replace([np.inf, -np.inf], np.nan)
    print(f"{role}: 유효 {values.notna().sum()} / 전체 {len(values)}")
    return values

## Section 3. Basic Statistics

실제 존재하는 모든 numeric variable의 count, mean, std, min/max, quantile을 확인합니다.
`count=0`이면 유효한 관측이 없으며 `std=NaN`인 소수 표본 그룹을 확정적으로 해석하지 않습니다.

In [7]:
if df is None:
    print("데이터 대기 중: 통계량을 계산하지 않습니다.")
elif not numeric_columns:
    print("수치 열이 없습니다. dtype과 CSV 설정을 확인하세요.")
else:
    numeric_data = df[numeric_columns].replace([np.inf, -np.inf], np.nan)
    print("유한값과 결측을 기준으로 요약합니다. 무한대는 결측으로 취급합니다.")
    display(numeric_data.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

데이터 대기 중: 통계량을 계산하지 않습니다.


## Section 4. Visualization

### 질문: ICBV 및 CapL은 어떤 분포를 가지고 있는가?

→ 각 측정값의 Histogram. 열이 없거나 유효값이 없으면 그림을 생략합니다.

In [8]:
def plot_distribution(data, role):
    values = numeric_measurement(data, role)
    if values is None or values.dropna().empty:
        print(f"{role}: 분포를 그릴 유효값이 없습니다.")
        return
    plt.figure(figsize=(7, 4))
    plt.hist(values.dropna(), bins=20, edgecolor="white")
    plt.title(f"{role} distribution (n={values.notna().sum()})")
    plt.xlabel(f"{role} (source units)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [9]:
plot_distribution(df, "ICBV")

ICBV: 데이터 또는 매핑된 열이 없어 생략합니다.
ICBV: 분포를 그릴 유효값이 없습니다.


### What do you observe?

1. 어떤 그룹의 분포가 다른가? (그룹이 없으면 분포의 모양을 적으세요.)
2. 이상치로 보이는 값이 있는가?
3. 변수 간 관계가 있어 보이는가? (단일 변수 그림이면 다음 비교를 제안하세요.)
4. 추가로 확인하고 싶은 것은 무엇인가?

Observation:
-
-
-

In [10]:
plot_distribution(df, "CapL")

CapL: 데이터 또는 매핑된 열이 없어 생략합니다.
CapL: 분포를 그릴 유효값이 없습니다.


### What do you observe?

1. 어떤 그룹의 분포가 다른가? (그룹이 없으면 분포의 모양을 적으세요.)
2. 이상치로 보이는 값이 있는가?
3. 변수 간 관계가 있어 보이는가? (단일 변수 그림이면 다음 비교를 제안하세요.)
4. 추가로 확인하고 싶은 것은 무엇인가?

Observation:
-
-
-

### 질문: ICBV와 CapL은 함께 변하는가?

→ 두 값이 **같은 행에서 모두 유효한** 관측만 Scatter plot에 사용합니다.
행 단위의 매칭이 적절한지는 현업 담당자에게 확인합니다.

In [11]:
def plot_relationship(data):
    icbv = numeric_measurement(data, "ICBV")
    capl = numeric_measurement(data, "CapL")
    if icbv is None or capl is None:
        return
    paired = pd.DataFrame({"ICBV": icbv, "CapL": capl}).dropna()
    if paired.empty:
        print("ICBV와 CapL이 모두 유효한 행이 없습니다.")
        return
    plt.figure(figsize=(7, 4))
    plt.scatter(paired["ICBV"], paired["CapL"], alpha=0.6)
    plt.title(f"ICBV vs CapL (paired n={len(paired)})")
    plt.xlabel("ICBV (source units)")
    plt.ylabel("CapL (source units)")
    plt.tight_layout()
    plt.show()

plot_relationship(df)

ICBV: 데이터 또는 매핑된 열이 없어 생략합니다.
CapL: 데이터 또는 매핑된 열이 없어 생략합니다.


### What do you observe?

1. 어떤 그룹의 분포가 다른가? (그룹이 없으면 분포의 모양을 적으세요.)
2. 이상치로 보이는 값이 있는가?
3. 변수 간 관계가 있어 보이는가? (단일 변수 그림이면 다음 비교를 제안하세요.)
4. 추가로 확인하고 싶은 것은 무엇인가?

Observation:
-
-
-

## Section 5. Context Analysis

**질문:** 전체에서 보인 패턴이 Context를 나누어도 유지되는가?

Overall → Product-wise → Equipment-wise → Recipe-wise → Equipment × Recipe

ICBV와 CapL을 각각 요약하며 mean·std·count를 함께 봅니다. Context 결측도 그룹으로 남깁니다.
Chamber가 장비 안에서만 유일한 이름이면 Equipment × Chamber로 비교합니다.

In [12]:
def context_summary(data, roles, target="CapL"):
    values = numeric_measurement(data, target)
    if values is None:
        return pd.DataFrame()
    if not roles:
        return values.agg(["count", "mean", "std", "min", "max"]).to_frame(target)
    columns = [column_map.get(role) for role in roles]
    if any(column is None or column not in data.columns for column in columns):
        print("Context 열이 없어 생략:", roles)
        return pd.DataFrame()
    target_column = column_map[target]
    working = data.copy()
    working[target_column] = values
    return working.groupby(columns, dropna=False)[target_column].agg(
        ["size", "count", "mean", "std", "min", "max"]
    )

In [13]:
contexts = [[], ["PRODUCT"], ["EQUIPMENT"], ["RECIPE"], ["EQUIPMENT", "RECIPE"]]
if df is not None:
    for target in ["ICBV", "CapL"]:
        for roles in contexts:
            print(target, "|", " × ".join(roles) if roles else "Overall")
            display(context_summary(df, roles, target))
    if column_map.get("CHAMBER"):
        chamber_roles = ["CHAMBER"]
        if column_map.get("EQUIPMENT") in df.columns:
            chamber_roles = ["EQUIPMENT", "CHAMBER"]
        display(context_summary(df, chamber_roles))
else:
    print("데이터 수령 후 Overall부터 순서대로 비교합니다.")

데이터 수령 후 Overall부터 순서대로 비교합니다.


**결과 확인:** `size - count`는 해당 그룹의 타깃 결측/무한대 수입니다.
그룹별 평균 차이는 표본 수·산포·Product 구성을 함께 고려합니다.
장비 안에서도 Recipe마다 차이가 달라지는지 확인한 뒤 원인 설명을 논의하세요.

### 질문: Context별 CapL 분포와 산포가 다른가?

→ 아래 `CONTEXT_ROLE`로 Product/Equipment/Recipe Boxplot을 선택합니다.
범주가 많으면 현업과 비교 대상을 정한 후 별도로 filtering하세요.

In [14]:
def plot_context(data, role, target="CapL"):
    values = numeric_measurement(data, target)
    column = column_map.get(role)
    if values is None or column is None or column not in data.columns:
        print("선택한 Context를 비교할 수 없습니다.")
        return
    plot_data = pd.DataFrame({"Context": data[column], "Value": values}).dropna()
    if plot_data.empty or plot_data["Context"].nunique() > 20:
        print("유효값이 없거나 범주가 20개를 초과합니다. 비교 대상을 확인하세요.")
        return
    print("그림 유효 행:", len(plot_data), "/ 전체:", len(data))
    plot_data.boxplot(column="Value", by="Context", figsize=(8, 4), rot=30)
    plt.suptitle("")
    plt.title(f"{target} by {role}")
    plt.xlabel(role)
    plt.ylabel(f"{target} (source units)")
    plt.tight_layout()
    plt.show()

In [15]:
CONTEXT_ROLE = "EQUIPMENT"  # "PRODUCT" 또는 "RECIPE"로 바꿔 비교합니다.
plot_context(df, CONTEXT_ROLE)

CapL: 데이터 또는 매핑된 열이 없어 생략합니다.
선택한 Context를 비교할 수 없습니다.


### What do you observe?

1. 어떤 그룹의 분포가 다른가? (그룹이 없으면 분포의 모양을 적으세요.)
2. 이상치로 보이는 값이 있는가?
3. 변수 간 관계가 있어 보이는가? (단일 변수 그림이면 다음 비교를 제안하세요.)
4. 추가로 확인하고 싶은 것은 무엇인가?

Observation:
-
-
-

## Section 6. Observation

## Current Observation

고려대 연구진이 작성합니다. 실제 데이터가 없으면 미작성으로 둡니다.
관찰마다 **비교 조건 / 유효 표본 수 / 근거 표 또는 그림 / 불확실한 점**을 함께 적으세요.

Observation 1:

Observation 2:

Observation 3:

## Section 7. Engineer Discussion — Discuss

## Discussion with Process Engineers

### Q1.
특정 Equipment에서 분포가 다른 이유가 실제 공정 특성과 관련되어 있나요?

Engineer Comment:
______________________________________

### Q2.
Product 또는 Recipe마다 정상적인 ICBV 수준이 달라질 수 있나요?

Engineer Comment:
______________________________________

### Q3.
현재 데이터만으로는 확인하기 어려운 추가적인 공정 Context가 있나요?

Engineer Comment:
______________________________________

추가 확인: Chamber 명칭의 유일성, 측정 시점 차이, 재측정 이력, 표본 선택 기준.

## Section 8. Domain Knowledge Injection

현업 설명을 검증 가능한 ML hypothesis로 바꿉니다.

Domain Knowledge:

→

ML Hypothesis:

→

Possible Feature / Analysis:

**작성 예시(실제 관찰이 아님):**
특정 Recipe는 Equipment 상태에 따라 영향이 다르다
→ Equipment에 따라 Recipe와 CapL의 관계가 다를 수 있다
→ Equipment × Recipe interaction을 분석하고 Product 구성이 같은지 확인한다.

- 이 가설을 지지/반박할 결과:
- 필요한 추가 데이터 / 현업 확인:
- 제안한 Feature는 실제 예측 시점에도 얻을 수 있는가?:

## Section 9. Improve / Additional Analysis

**질문 예시:** 특정 Product로 범위를 맞춘 뒤에도 Equipment × Recipe별 차이가 남는가?

아래는 실행 가능한 분석 skeleton입니다. `SELECTED_PRODUCT`를 실제 값으로 정해야 비교가 수행됩니다.
값을 선택하기 전에는 결과를 만들지 않습니다. 원본을 보존하고 분석용 subset을 만듭니다.

In [16]:
SELECTED_PRODUCT = None  # 실제 PRODUCT 값으로 수정합니다.
baseline_summary = pd.DataFrame()
improved_summary = pd.DataFrame()
analysis_df = None
product_column = column_map.get("PRODUCT")
if df is None:
    print("실제 데이터가 필요합니다.")
elif SELECTED_PRODUCT is None:
    print("Engineer Discussion 후 SELECTED_PRODUCT를 지정하세요.")
elif product_column is None or product_column not in df.columns:
    print("PRODUCT column mapping을 확인하세요.")
else:
    analysis_df = df[df[product_column] == SELECTED_PRODUCT].copy()
    if analysis_df.empty:
        print("선택한 Product에 해당하는 행이 없습니다.")
    else:
        baseline_summary = context_summary(df, ["EQUIPMENT", "RECIPE"])
        improved_summary = context_summary(analysis_df, ["EQUIPMENT", "RECIPE"])
        print("전체 행:", len(df), "| 선택 Product 행:", len(analysis_df))

실제 데이터가 필요합니다.


In [17]:
# TODO: 현업 의견에서 도출한 새 가설을 분석하세요.
# 예: 동일 Product에서 Chamber별 차이, 결측 여부에 따른 Context 구성 비교.
# 원본 df는 보존하고 분석 조건과 유효 표본 수를 함께 기록하세요.

## Section 10. Re-Analyze

이전(전체 데이터)과 추가 분석(선택 Product)을 나란히 비교합니다.
표본 구성이 달라진 비교이므로 차이가 줄어도 모델 성능 개선이나 인과 효과로 해석하지 않습니다.

In [18]:
if baseline_summary.empty or improved_summary.empty:
    print("전후 비교 대기: Section 9에서 실제 분석 조건을 먼저 지정하세요.")
else:
    comparison = pd.concat(
        {"Before: all products": baseline_summary,
         "After: selected product": improved_summary}, axis=1,
    )
    display(comparison)

전후 비교 대기: Section 9에서 실제 분석 조건을 먼저 지정하세요.


| 비교 항목 | 기존 분석 | 추가 분석 | 해석 / 남은 한계 |
|---|---|---|---|
| 대상 범위 / 유효 표본 수 | | | |
| 그룹 평균 / 산포 | | | |
| 가설을 지지 또는 반박하는 근거 | | | |

- 반대 설명으로 가능한 것은?:
- 같은 기준으로 다시 확인할 분석은?:

## Section 11. Conclusion & Next Hypothesis

오늘의 관찰과 현업 의견을 구분해 기록합니다. 다음 Hypothesis에는 비교할 조건과 필요한 데이터를 적어 다음 회차의 출발점으로 사용하세요.

# Conclusion

## What we learned
1.
2.
3.

## What we still don't know
1.
2.
3.

## Next Hypothesis
H1.

H2.